In [256]:
gold_cfg={}

In [257]:
gen_cfg_gpt= {}

In [258]:
gen_cfg_claude = {}

In [259]:
gen_cfg_gemini = {}

In [260]:
gen_cfg_qwen = {}

更新extract test path方法, add nGED and k-path-overlap

In [6]:
import numpy as np
import networkx as nx
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from collections import Counter
import matplotlib.pyplot as plt
import re

class CFGComparator:
    def __init__(self, model_name='all-mpnet-base-v2', sim_threshold=0.75):
        # 调低硬阈值至 0.65，因为我们会用“包含度”逻辑来弥补
        self.model = SentenceTransformer(model_name)
        self.sim_threshold = sim_threshold
        self.text_cache = {}

    def _encode_batch(self, texts):
        unique_texts = list(set(texts))
        needed = [t for t in unique_texts if t not in self.text_cache]
        if needed:
            embeddings = self.model.encode(needed, show_progress_bar=False)
            for text, emb in zip(needed, embeddings):
                self.text_cache[text] = emb
        return np.array([self.text_cache[t] for t in texts])

    def _calculate_containment_score(self, gold_text, gen_text, cosine_sim):
        """
        计算包含度分数：判断 gold 是否在语义上被 gen 包含。
        """
        g = gold_text.lower()
        c = gen_text.lower()
        
        # 1. 基础清理：去除干扰匹配的控制引导词
        g_clean = re.sub(r'\b(if|then|else|step|system|the)\b', '', g).strip()
        c_clean = re.sub(r'\b(if|then|else|step|system|the)\b', '', c).strip()

        # 2. 关键词包含检查 (简易 Jaccard)
        g_words = set(re.findall(r'\w+', g_clean))
        c_words = set(re.findall(r'\w+', c_clean))
        if not g_words: return cosine_sim
        
        intersection = g_words.intersection(c_words)
        containment_ratio = len(intersection) / len(g_words)
        
        # 3. 综合评分：如果关键词包含度很高，则大幅拉升相似度分数
        # 这样即便 cosine 只有 0.6，只要关键词全对，分数就能过阈值
        combined_score = (cosine_sim * 0.4) + (containment_ratio * 0.6)
        return combined_score

    def _preprocess_cfg(self, cfg):
        G = nx.DiGraph()
        for node in cfg["nodes"]:
            G.add_node(node["id"], statement=node["Statement"])
        for edge in cfg["edges"]:
            G.add_edge(edge["from"], edge["to"], condition=edge.get("condition", ""))
        return G

    def compare_nodes(self, gold_cfg, gen_cfg):
        gold_nodes = {node["id"]: node["Statement"] for node in gold_cfg["nodes"]}
        print("gold_nodes", len(gold_nodes))
        gen_nodes = {node["id"]: node["Statement"] for node in gen_cfg["nodes"]}
        print("gen_nodes", len(gen_nodes))
        gold_ids, gen_ids = list(gold_nodes.keys()), list(gen_nodes.keys())
        
        gold_embs = self._encode_batch(list(gold_nodes.values()))
        gen_embs = self._encode_batch(list(gen_nodes.values()))

        # 初始余弦相似度矩阵
        base_sim_matrix = cosine_similarity(gold_embs, gen_embs)
        
        # 注入包含度逻辑
        final_sim_matrix = np.zeros_like(base_sim_matrix)
        for i, g_id in enumerate(gold_ids):
            for j, c_id in enumerate(gen_ids):
                final_sim_matrix[i, j] = self._calculate_containment_score(
                    gold_nodes[g_id], gen_nodes[c_id], base_sim_matrix[i, j]
                )

        row_ind, col_ind = linear_sum_assignment(-final_sim_matrix)

        mapping_thresh = {}
        full_mapping = {}
        matched_pairs = []

        for i, j in zip(row_ind, col_ind):
            sim = final_sim_matrix[i, j]
            g_id, c_id = gold_ids[i], gen_ids[j]
            full_mapping[c_id] = g_id
            if sim >= self.sim_threshold:
                mapping_thresh[c_id] = g_id
                matched_pairs.append((g_id, c_id))

        tp = len(matched_pairs)
        return {
            "precision": tp / len(gen_ids) if gen_ids else 0,
            "recall": tp / len(gold_ids) if gold_ids else 0,
            "f1": (2 * tp) / (len(gen_ids) + len(gold_ids)) if (len(gen_ids) + len(gold_ids)) else 0,
            "mapping": mapping_thresh,
            "full_mapping": full_mapping,
            "matched_pairs": matched_pairs
        }

    def compare_edges(self, gold_cfg, gen_cfg, node_results):
        """
        边评估：支持冗余惩罚、安全命名空间、并返回详细的差异列表。
        """
        G_gold = self._preprocess_cfg(gold_cfg)
        # 使用通过相似度阈值筛选后的有效映射
        valid_map = node_results["mapping"] 
    
        # --- 1. 节点处理 (List 形式保留冗余信息) ---
        nodes_gold = list(G_gold.nodes())
        nodes_gen_mapped = []
        for n in [node["id"] for node in gen_cfg["nodes"]]:
            if n in valid_map:
                nodes_gen_mapped.append(valid_map[n])
            else:
                # 使用更安全的命名空间，防止与 Gold ID 碰撞
                nodes_gen_mapped.append(f"GEN_UNMATCHED::{n}")
    
        # --- 2. 边处理 (List 形式保留冗余信息) ---
        edges_gold = list(G_gold.edges())
        edges_gen_mapped = []
        for u, v in [ (e["from"], e["to"]) for e in gen_cfg["edges"] ]:
            new_u = valid_map.get(u, f"GEN_UNMATCHED::{u}")
            new_v = valid_map.get(v, f"GEN_UNMATCHED::{v}")
            edges_gen_mapped.append((new_u, new_v))
    
        # --- 3. 基于多重集(Multiset)的差异计算 ---
        c_nodes_gold = Counter(nodes_gold)
        c_nodes_gen = Counter(nodes_gen_mapped)
        c_edges_gold = Counter(edges_gold)
        c_edges_gen = Counter(edges_gen_mapped)
    
        # 计算节点和边的编辑代价 (考虑了多对一和冗余)
        node_diff = sum((c_nodes_gold - c_nodes_gen).values()) + sum((c_nodes_gen - c_nodes_gold).values())
        edge_diff = sum((c_edges_gold - c_edges_gen).values()) + sum((c_edges_gen - c_edges_gold).values())
        
        # 交集大小 (True Positives)
        tp_edges_set = c_edges_gold & c_edges_gen
        tp_count = sum(tp_edges_set.values())
        
        # --- 4. 指标计算 ---
        edge_precision = tp_count / len(edges_gen_mapped) if edges_gen_mapped else 0
        edge_recall = tp_count / len(edges_gold) if edges_gold else 0
        edge_f1 = (2 * edge_precision * edge_recall / (edge_precision + edge_recall)) if (edge_precision + edge_recall) > 0 else 0
        
        # max_size = max(len(nodes_gold) + len(edges_gold), len(nodes_gen_mapped) + len(edges_gen_mapped))
        max_size = (len(nodes_gold) + len(edges_gold)) + (len(nodes_gen_mapped) + len(edges_gen_mapped))
        nGED = (node_diff + edge_diff) / max_size if max_size > 0 else 0.0
    
        # --- 5. 差异列表 (转换为 set 方便打印查看唯一差异) ---
        # 注意：这里转 set 仅为了输出列表，计算 nGED 时已经考虑了重复项
        set_edges_gold = set(edges_gold)
        set_edges_gen = set(edges_gen_mapped)
    
        return {
            "edge_precision": edge_precision,
            "edge_recall": edge_recall,
            "edge_f1": edge_f1,
            "nGED": nGED,
            "tp_count": tp_count,
            "missing_edges": list(set_edges_gold - set_edges_gen),
            "extra_edges": list(set_edges_gen - set_edges_gold)
        }

    # ---------------------------
    # 4. 可视化分析
    # ---------------------------
    def visualize_comparison(self, gold_cfg, gen_cfg, node_results):
        G_gold = self._preprocess_cfg(gold_cfg)
        G_gen = self._preprocess_cfg(gen_cfg)
        matched_ids = {g_id for g_id, c_id in node_results["matched_pairs"]}
        gen_matched_ids = {c_id for g_id, c_id in node_results["matched_pairs"]}

        plt.figure(figsize=(16, 8))

        # Gold 标准图
        plt.subplot(121)
        pos1 = nx.spring_layout(G_gold, seed=42)
        colors1 = ['lightgreen' if n in matched_ids else 'salmon' for n in G_gold.nodes()]
        nx.draw(G_gold, pos1, with_labels=True, node_color=colors1, node_size=800, font_size=8)
        plt.title("Gold CFG (Green = Matched)")

        # 生成的图
        plt.subplot(122)
        pos2 = nx.spring_layout(G_gen, seed=42)
        colors2 = ['lightblue' if n in gen_matched_ids else 'orange' for n in G_gen.nodes()]
        nx.draw(G_gen, pos2, with_labels=True, node_color=colors2, node_size=800, font_size=8)
        plt.title("Generated CFG (Blue = Matched)")

        plt.tight_layout()
        plt.show()


In [261]:
comparator = CFGComparator()

gen_cfgs_list = [
    ("GPT", gen_cfg_gpt),
    ("Claude", gen_cfg_claude),
    ("Gemini", gen_cfg_gemini),
    ("Qwen", gen_cfg_qwen)
]

for i, (method_name, gen_cfg) in enumerate(gen_cfgs_list):
    print(f"\n{'='*30}")
    print(f"正在评估第 {i+1}/{len(gen_cfgs_list)} 个方法: {method_name}")
    print(f"{'='*30}")

    try:
        # 1. 节点对齐
        node_res = comparator.compare_nodes(gold_cfg, gen_cfg)
        
        # 2. 边与结构对齐 (nGED)
        edge_res = comparator.compare_edges(gold_cfg, gen_cfg, node_res)
        
        print("-" * 30)
        print(f"节点评估: Precision={node_res['precision']:.2f}, Recall={node_res['recall']:.2f}, F1={node_res['f1']:.2f}")
        print(f"边评估:   Precision={edge_res['edge_precision']:.2f}, Recall={edge_res['edge_recall']:.2f}, F1={edge_res['edge_f1']:.2f}")
        print(f"归一化图编辑距离 (nGED): {edge_res['nGED']:.4f}")
        print("-" * 30)


    except Exception as e:
        print(f"❌ 评估 {method_name} 时出错: {e}")
        continue


正在评估第 1/4 个方法: GPT
gold_nodes 8
gen_nodes 8
------------------------------
节点评估: Precision=1.00, Recall=1.00, F1=1.00
边评估:   Precision=1.00, Recall=1.00, F1=1.00
归一化图编辑距离 (nGED): 0.0000
------------------------------

正在评估第 2/4 个方法: Claude
gold_nodes 8
gen_nodes 8
------------------------------
节点评估: Precision=1.00, Recall=1.00, F1=1.00
边评估:   Precision=1.00, Recall=1.00, F1=1.00
归一化图编辑距离 (nGED): 0.0000
------------------------------

正在评估第 3/4 个方法: Gemini
gold_nodes 8
gen_nodes 8
------------------------------
节点评估: Precision=1.00, Recall=1.00, F1=1.00
边评估:   Precision=1.00, Recall=1.00, F1=1.00
归一化图编辑距离 (nGED): 0.0000
------------------------------

正在评估第 4/4 个方法: Qwen
gold_nodes 8
gen_nodes 8
------------------------------
节点评估: Precision=0.88, Recall=0.88, F1=0.88
边评估:   Precision=0.67, Recall=0.75, F1=0.71
归一化图编辑距离 (nGED): 0.2121
------------------------------
